# 04 · Manager and Workers

The router sends a message to **one** specialist. But some questions need **several**:

> "My order A1001 was delivered a while ago. Can I still return it, and how long until I get my money back?"

That needs the **Orders agent** (when was it delivered?) **and** the **Policy agent** (what is the return window, how are refunds paid?).

Pattern 2: a **manager** agent that does not know anything itself. Its tools are its **colleagues**. It asks each specialist a question, then combines the answers.

```
Customer ──> Manager ──asks──> Orders agent
                    └──asks──> Policy agent
                    <── combines the answers ──
```

In [1]:
import json
from shop import client, MODEL     # the OpenAI client and the model name (see shop.py)

print("Ready")

Ready


In [2]:
from shop import run_agent, orders_agent, policy_agent, make_tool

## Specialists as tools
Remember: a tool is just a Python function plus a description. So we can write a function that **runs a specialist agent**, and give it to the manager as a tool.

In [3]:
def ask_orders_agent(question):
    """Ask the Orders specialist a question."""
    return run_agent(orders_agent, question)

def ask_policy_agent(question):
    """Ask the Policy specialist a question."""
    return run_agent(policy_agent, question)

ASK_ORDERS_TOOL = make_tool(
    "ask_orders_agent",
    "Ask the Orders specialist about a customer's order. Include the order id in the question.",
    "question", "A clear question about one order")

ASK_POLICY_TOOL = make_tool(
    "ask_policy_agent",
    "Ask the Policy specialist about shop rules. Ask about one topic at a time.",
    "question", "A clear question about shop rules")

## The manager
Its instructions say it must **delegate**: it has no way to look anything up itself.

In [4]:
manager_agent = {
    "name": "Manager",
    "instructions": (
        "You are a support manager at an online shop. You cannot look anything up yourself. "
        "Use ask_orders_agent only to get facts about an order, such as the item, its status and its delivery date. "
        "Use ask_policy_agent for questions about shop rules. "
        "Ask each colleague one clear question at a time, and decide yourself how the facts and the rules fit together. "
        "Then combine what they told you into one reply to the customer of at most 3 sentences."
    ),
    "tools": [ASK_ORDERS_TOOL, ASK_POLICY_TOOL],
    "functions": {"ask_orders_agent": ask_orders_agent, "ask_policy_agent": ask_policy_agent},
}

## Watch the manager work
Lines in square brackets show **who is doing what**. The manager's lines come first, then each specialist's lines.

In [5]:
question = "My order A1001 was delivered a while ago. Can I still return it, and how long until I get my money back?"

print("Customer:", question, "\n")
answer = run_agent(manager_agent, question)

print("\nManager's reply:", answer)

Customer: My order A1001 was delivered a while ago. Can I still return it, and how long until I get my money back? 



  [Manager] ask_orders_agent({'question': 'What is the status of order A1001?'})


  [Orders agent] lookup_order({'order_id': 'A1001'})


  [Manager] ask_policy_agent({'question': 'What is the return policy and how long until a customer receives their money back after returning an order?'})


  [Policy agent] search_policies({'question': 'return policy'})


  [Policy agent] search_policies({'question': 'refund timeline'})
  [Policy agent] search_policies({'question': 'return and refund'})



Manager's reply: Your order A1001 has been delivered, and since it was delivered on November 3, 2026, you can still return it as you are within the 30-day return window. Once we receive the returned item, you can expect your refund to be processed back to your original payment method within 5 to 7 business days.


The manager asked the **Orders agent** for the delivery date, and the **Policy agent** for the return window and how refunds are paid. Then it combined them into one answer.

## Router or manager?

| | Router (notebook 03) | Manager (this notebook) |
|---|---|---|
| Specialists used | **One** per message | **Several** per message |
| Who decides | A separate routing step | The manager, using its tools |
| Best for | Clear, single-topic messages | Questions that mix topics |
| Cost | Fewer model calls | More model calls |

## Recap
* A **manager** agent uses other agents as **tools**. It delegates and combines.
* Nothing new is needed: an agent used as a tool is just a function that calls `run_agent`.
* More agents means **more model calls**. Use a manager only when a question really needs it.

Next: a pattern for **quality**. Have one agent check another agent's work.